## End-to-End ML Platform for Financial Fraud / Risk Scoring


# Import libraries

In [5]:
import pandas as pd
import numpy as np
import os, kagglehub
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

# Load dataset

In [2]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(os.path.join(path, "creditcard.csv"))

In [3]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# Performing Exploratory Data Analysis(EDA) to understand structure and quality of data

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

# Checking the value count of class that refers as 1 as fraud transaction and 0 as Legit transaction

In [5]:
data["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

This data is highly unbalanced,
seperating data to do analysis

In [6]:
legit = data[data.Class == 0]
fraud = data[data.Class == 1]

In [7]:
print(legit.shape)
print(fraud.shape)

(284315, 31)
(492, 31)


In [8]:
legit.Amount.describe()

count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

In [9]:
fraud.Amount.describe()

count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

In [10]:
fraud.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
541,406.0,-2.312227,1.951992,-1.609851,3.997906,-0.522188,-1.426545,-2.537387,1.391657,-2.770089,...,0.517232,-0.035049,-0.465211,0.320198,0.044519,0.177840,0.261145,-0.143276,0.00,1
623,472.0,-3.043541,-3.157307,1.088463,2.288644,1.359805,-1.064823,0.325574,-0.067794,-0.270953,...,0.661696,0.435477,1.375966,-0.293803,0.279798,-0.145362,-0.252773,0.035764,529.00,1
4920,4462.0,-2.303350,1.759247,-0.359745,2.330243,-0.821628,-0.075788,0.562320,-0.399147,-0.238253,...,-0.294166,-0.932391,0.172726,-0.087330,-0.156114,-0.542628,0.039566,-0.153029,239.93,1
6108,6986.0,-4.397974,1.358367,-2.592844,2.679787,-1.128131,-1.706536,-3.496197,-0.248778,-0.247768,...,0.573574,0.176968,-0.436207,-0.053502,0.252405,-0.657488,-0.827136,0.849573,59.00,1
6329,7519.0,1.234235,3.019740,-4.304597,4.732795,3.624201,-1.357746,1.713445,-0.496358,-1.282858,...,-0.379068,-0.704181,-0.656805,-1.632653,1.488901,0.566797,-0.010016,0.146793,1.00,1


In [11]:
data.groupby("Class").mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


As we can see there is huge difference between the mean of legit data and fraud data so doing unsampling by distributing dataset randomly into equal number for both the transaction

In [12]:
legit_sample = legit.sample(n=492)

Now Concatenate both the data into one dataset by using concat from pandas

In [13]:
new_data = pd.concat([legit_sample, fraud], axis=0) # axis = 0(row) and axis = 1(column)

In [14]:
new_data.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
94898,65078.0,-0.431913,0.860821,2.613309,1.720322,-0.816285,0.019871,-0.048712,0.070440,-0.002135,...,0.083748,0.485420,-0.099761,0.746049,-0.336068,-0.235349,0.086699,0.156879,9.99,0
199152,132829.0,2.142196,-0.806359,-1.098418,-0.578882,-0.454845,-0.388197,-0.479183,-0.181802,-0.240303,...,-0.899241,-2.127628,0.456800,-1.084530,-0.666192,0.101970,-0.041618,-0.047214,43.90,0
163453,115949.0,0.127567,0.570221,-0.077807,-0.508839,-0.149237,0.281867,-0.707469,0.851477,0.552183,...,0.135838,0.293470,0.252590,0.167053,-1.266395,0.185495,-0.045375,-0.011609,2.45,0
240340,150560.0,1.987104,-0.327904,-0.264387,0.472372,-0.784915,-0.778532,-0.496184,-0.054496,1.376857,...,-0.188223,-0.414125,0.383511,-0.016141,-0.450694,-0.622417,0.031415,-0.032308,5.00,0
136558,81760.0,0.408190,-1.438129,0.065753,0.905215,-0.964679,-0.040028,0.231720,0.030333,0.449034,...,-0.036791,-0.973749,-0.321480,0.007351,0.134343,0.249900,-0.113791,0.071179,420.08,0


In [15]:
new_data["Class"].value_counts()

Class
0    492
1    492
Name: count, dtype: int64

In [16]:
new_data.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,95986.619919,0.052563,0.048215,-0.055883,0.041665,-0.087674,-0.003221,0.002242,0.020670,-0.088656,...,0.022061,-0.030566,-0.030585,-0.011757,0.008666,-0.021265,0.024337,-0.015280,0.007364,94.792317
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


Split the dataset into X and Y for classification

In [17]:
X = new_data.drop(columns="Class", axis=1)
Y = new_data["Class"]

In [18]:
X

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
94898,65078.0,-0.431913,0.860821,2.613309,1.720322,-0.816285,0.019871,-0.048712,0.070440,-0.002135,...,0.156172,0.083748,0.485420,-0.099761,0.746049,-0.336068,-0.235349,0.086699,0.156879,9.99
199152,132829.0,2.142196,-0.806359,-1.098418,-0.578882,-0.454845,-0.388197,-0.479183,-0.181802,-0.240303,...,-0.496128,-0.899241,-2.127628,0.456800,-1.084530,-0.666192,0.101970,-0.041618,-0.047214,43.90
163453,115949.0,0.127567,0.570221,-0.077807,-0.508839,-0.149237,0.281867,-0.707469,0.851477,0.552183,...,-0.291923,0.135838,0.293470,0.252590,0.167053,-1.266395,0.185495,-0.045375,-0.011609,2.45
240340,150560.0,1.987104,-0.327904,-0.264387,0.472372,-0.784915,-0.778532,-0.496184,-0.054496,1.376857,...,-0.273876,-0.188223,-0.414125,0.383511,-0.016141,-0.450694,-0.622417,0.031415,-0.032308,5.00
136558,81760.0,0.408190,-1.438129,0.065753,0.905215,-0.964679,-0.040028,0.231720,0.030333,0.449034,...,0.671019,-0.036791,-0.973749,-0.321480,0.007351,0.134343,0.249900,-0.113791,0.071179,420.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,1.252967,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.226138,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.247968,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.306271,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00


In [19]:
Y

94898     0
199152    0
163453    0
240340    0
136558    0
         ..
279863    1
280143    1
280149    1
281144    1
281674    1
Name: Class, Length: 984, dtype: int64

Using train-test split from SKlearn for train and test

In [20]:
X_train, X_test , y_train, y_test = train_test_split(X, Y, test_size=0.2)

# Model Training

Lets create a pipeline for three different model and compare them.

In [21]:
Models = {
    "RandomForest": RandomForestClassifier(n_estimators= 200, random_state= 42),
    "XGBoost": xgb.XGBClassifier(eval_metric = ["logloss", "auc"] ,n_estimators= 200, random_state= 42),
    "LightGBM": lgb.LGBMClassifier(n_estimators= 200, random_state= 42, verbosity = -1)
}

In [22]:
Results = {}

for name, model in Models.items():
  # train model
    model.fit(X_train, y_train)

    # Get predicted probabilities
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Convert probabilities to binary predictions
    y_train_pred = (y_train_prob >= 0.5).astype(int)
    y_test_pred = (y_test_prob >= 0.5).astype(int)

    # Metrics
    roc_train = float(roc_auc_score(y_train, y_train_prob))
    roc_test = float(roc_auc_score(y_test, y_test_prob))
    precision = float(precision_score(y_test, y_test_pred))
    recall = float(recall_score(y_test, y_test_pred))
    accuracy = float(accuracy_score(y_test, y_test_pred))

  # check overfit/ underfit
    if roc_train - roc_test > 0.1:
        fit_status = "Overfit"
    elif roc_train < 0.6 and roc_test < 0.6:
        fit_status = "Underfit"
    else:
        fit_status = "Good fit"

    Results[name] = {
        "model": model,
         "metrics": {
            "roc_train": roc_train,
            "roc_test": roc_test,
            "precision": precision,
            "recall": recall,
            "accuracy": accuracy
        },
        "fit_status": fit_status
    }

In [23]:
Result_df = pd.DataFrame({
    model_name: {**metrics_info['metrics'], 'fit_status': metrics_info['fit_status']}
    for model_name, metrics_info in Results.items()
}).T

# Ensure numeric columns are float
numeric_cols = ['roc_train', 'roc_test', 'precision', 'recall', 'accuracy']
Result_df[numeric_cols] = Result_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Show comparison table
print("\n== Model Comparison ==")
print(Result_df)


== Model Comparison ==
              roc_train  roc_test  precision    recall  accuracy fit_status
RandomForest        1.0  0.990269   0.976471  0.902174  0.944162   Good fit
XGBoost             1.0  0.985300   0.954023  0.902174  0.934010   Good fit
LightGBM            1.0  0.987578   0.976744  0.913043  0.949239   Good fit


In [24]:
best_model = Result_df['roc_test'].idxmax()
best_model_score = Result_df.loc[best_model, 'roc_test']

print(f"\nBest Model: {best_model} with ROC Test Score: {best_model_score:.4f}")


Best Model: RandomForest with ROC Test Score: 0.9903


In [25]:
best_model_save = Models[best_model]
print(best_model_save)

RandomForestClassifier(n_estimators=200, random_state=42)


# Save the model using joblib

In [26]:
file_path = f"C:/Users/Ronak/OneDrive/Desktop/Credit/models/{best_model}.joblib"
joblib.dump(best_model_save, file_path)

print(f"Model saved as: {file_path}")

Model saved as: C:/Users/Ronak/OneDrive/Desktop/Credit/RandomForest.joblib


In [27]:
#best_model_save = Models[best_model]
#joblib.dump(best_model, 'C:/Users/Ronak/OneDrive/Desktop/Credit/Fraud_scoring.pkl')

## Performing MLflow Experiment

In [28]:
Results

{'RandomForest': {'model': RandomForestClassifier(n_estimators=200, random_state=42),
  'metrics': {'roc_train': 1.0,
   'roc_test': 0.9902691511387163,
   'precision': 0.9764705882352941,
   'recall': 0.9021739130434783,
   'accuracy': 0.9441624365482234},
  'fit_status': 'Good fit'},
 'XGBoost': {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=False, eval_metric=['logloss', 'auc'],
                feature_types=None, feature_weights=None, gamma=None,
                grow_policy=None, importance_type=None,
                interaction_constraints=None, learning_rate=None, max_bin=None,
                max_cat_threshold=None, max_cat_to_onehot=None,
                max_delta_step=None, max_depth=None, max_leaves=None,
                min_child_weight=None, missing=nan, monotone_constraints=

In [29]:
print(type(Results))
print(Results)

<class 'dict'>
{'RandomForest': {'model': RandomForestClassifier(n_estimators=200, random_state=42), 'metrics': {'roc_train': 1.0, 'roc_test': 0.9902691511387163, 'precision': 0.9764705882352941, 'recall': 0.9021739130434783, 'accuracy': 0.9441624365482234}, 'fit_status': 'Good fit'}, 'XGBoost': {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=['logloss', 'auc'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              

In [30]:
# dagshub setup

import dagshub
dagshub.init(repo_owner='ronak3618', repo_name='MLflow_daghub_demo', mlflow=True)

Accessing as ronak3618

Initialized MLflow to track repo "ronak3618/MLflow_daghub_demo"

Repository ronak3618/MLflow_daghub_demo initialized!

In [31]:
mlflow.end_run()  

In [32]:
# os.environ['MLFLOW_TRACKING_USERNAME'] = 'ronak3618'
# os.environ['MLFLOW_TRACKING_PASSWORD'] = '6e89bf0a7ef54d9e6605276a0519635d2352c412'
# os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow'     !USE ALL THIS IF THE CODE SHOWS AN ERROR! 

# mlflow.set_tracking_uri(uri = "http://127.0.0.1:5000/") Commented as i am not running it on dagshub server
mlflow.set_experiment("Creditcardfraud_Models")

for model_name, data in Results.items():

    with mlflow.start_run(run_name=model_name):

        # Log model
        mlflow.sklearn.log_model(
            sk_model=data["model"],
            artifact_path="model"
        )

        # Log metrics
        for metric_name, metric_value in data["metrics"].items():
            mlflow.log_metric(metric_name, metric_value)

        # Log tags
        mlflow.set_tag("fit_status", data["fit_status"])
        mlflow.set_tag("model_name", model_name)
print("DONE")

2026/02/23 15:43:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 15:43:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0/runs/0d619c37681d41108aeb94be5a073f60
🧪 View experiment at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0


2026/02/23 15:44:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 15:44:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0/runs/349d0e3d53ee4e62a0d5035c63460fd1
🧪 View experiment at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0


2026/02/23 15:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 15:44:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0/runs/805a2f7e268a403bb231014af3d01491
🧪 View experiment at: https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow/#/experiments/0
DONE


In [33]:
print(mlflow.get_tracking_uri())

https://dagshub.com/ronak3618/MLflow_daghub_demo.mlflow


## Register the model in MLflow

In [ ]:
#model_name = 'LightGBM'
#run_id = input("Enter Run ID")
#model_uri = f"runs:/{run_id}/model"

#final = mlflow.register_model(
    #model_uri, 
    #model_name)

## Load the model

In [35]:
model_name = 'RandomForest'
model_version = 1
model_uri = f"models:/{model_name}/latest"  # models:/<YOUR_MODEL_NAME>/<YOUR_MODEL_VERSION>

loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

RestException: RESOURCE_DOES_NOT_EXIST: Response: {'error_code': 'RESOURCE_DOES_NOT_EXIST'}

In [38]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = "credit-card-fraud-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri = dev_model_uri , dst_name = prod_model)

Successfully registered model 'credit-card-fraud-prod'.
Copied version '1' of model 'LightGBM' to version '1' of model 'credit-card-fraud-prod'.


<ModelVersion: aliases=[], creation_timestamp=1770695610744, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1770695610744, metrics=None, model_id=None, name='credit-card-fraud-prod', params=None, run_id='16aaca8c7a9141d4a7489de43d94bcf8', run_link='', source='models:/104199_LightGBM/1', status='READY', status_message=None, tags={}, user_id='', version='1'>